# Eval 구축
Claude가 작업에서 가능한 한 높은 정확도를 제공하도록 최적화하는 것은 경험적 과학이며 지속적인 개선 과정입니다. 프롬프트 변경이 핵심 지표에서 model 성능을 개선했는지 확인하려는 경우든, model이 production에 출시할 만큼 충분히 좋은지 판단하려는 경우든, 좋은 offline evaluation 시스템은 성공에 필수입니다.

이 recipe에서는 평가 구축의 일반적인 패턴과 그때 따를 만한 유용한 경험칙을 살펴봅니다.

## Eval의 구성 요소
Eval은 일반적으로 네 부분으로 구성됩니다.
- model에 제공하는 입력 프롬프트. 이 프롬프트를 바탕으로 completion을 생성하도록 Claude에 요청합니다. eval을 설계할 때 input column에는 테스트 시 프롬프트 template에 제공하는 가변 입력 집합이 들어 있는 경우가 많습니다.
- 평가하려는 model에 입력 프롬프트를 실행해 얻은 출력.
- model 출력과 비교하는 "golden answer". golden answer는 반드시 정확히 일치해야 하는 답일 수도 있고, grader가 채점의 비교 기준으로 삼을 완벽한 답변 예제일 수도 있습니다.
- 아래에서 설명하는 grading 방법 중 하나로 생성하며, model이 질문에 얼마나 잘 답했는지 나타내는 점수.

## Eval Grading 방법
Eval에는 시간과 비용이 많이 들 수 있는 두 가지가 있습니다. 첫째는 eval의 질문과 golden answer 작성입니다. 둘째는 grading입니다. 사용할 dataset이 이미 없거나 질문을 수동으로 생성하지 않고 dataset을 만들 방법이 없으면(Claude로 질문을 생성하는 것을 고려하세요!) 질문과 golden answer 작성에 상당한 시간이 들 수 있지만, 일반적으로 한 번의 고정 비용이라는 이점이 있습니다. 질문과 golden answer를 작성하면 다시 작성해야 하는 경우는 매우 드뭅니다. 반면 grading은 eval을 다시 실행할 때마다 계속 발생하는 비용이며, eval은 많이 다시 실행하게 될 것입니다. 따라서 빠르고 저렴하게 평가할 수 있는 eval 구축이 설계 선택의 중심에 있어야 합니다.

Eval을 평가하는 일반적인 방법은 세 가지입니다.
- **Code-based grading:** 일반적인 코드(주로 string matching과 regular expression)를 사용해 model 출력을 평가합니다. 흔한 방식은 답변과의 정확한 일치를 확인하거나 문자열에 핵심 구절이 포함되어 있는지 확인하는 것입니다. 이를 허용하는 eval을 설계할 수 있다면 매우 빠르고 신뢰성이 높으므로 단연 최선의 grading 방법입니다. 하지만 많은 평가는 이런 grading 방식을 허용하지 않습니다.
- **Human grading:** 사람이 model이 생성한 답변을 보고 golden answer와 비교하여 점수를 부여합니다. 거의 모든 작업에 사용할 _수 있으므로_ 가장 능력 있는 grading 방법이지만, 특히 큰 eval을 구축했다면 매우 느리고 비용이 많이 들기도 합니다. 가능하다면 human grading이 필요한 eval 설계를 대체로 피하려고 해야 합니다.
- **Model-based grading:** Claude는 자신을 평가하는 능력이 매우 뛰어나며, 창작물의 어조 분석이나 자유 형식 질의응답의 정확도처럼 이전에는 사람이 필요했을 법한 다양한 작업의 grading에 사용할 수 있습니다. Claude용 _grader prompt_를 작성해 수행합니다.

각 grading 방법의 예제를 살펴보겠습니다.

### Code-based Grading
여기서는 어떤 대상의 다리 수를 Claude가 올바르게 식별하도록 요청하는 eval을 평가합니다. Claude가 다리 수만 출력하도록 하고, exact-match code-based grader를 사용할 수 있게 eval을 설계합니다.

In [ ]:
# 필요한 package 설치 및 불러오기, anthropic client 생성.
%pip install anthropic

In [2]:
from anthropic import Anthropic

client = Anthropic()
MODEL_NAME = "claude-opus-4-8"

In [6]:
# 작업의 입력 프롬프트 template 정의.
def build_input_prompt(animal_statement):
    user_content = f"""You will be provided a statement about an animal and your job is to determine how many legs that animal has.

    Here is the animal statment.
    <animal_statement>{animal_statement}</animal_statment>

    How many legs does the animal have? Return just the number of legs as an integer and nothing else."""

    messages = [{"role": "user", "content": user_content}]
    return messages

In [4]:
# eval 정의(실제로는 대신 jsonl 또는 csv 파일로 할 수 있음).
eval = [
    {"animal_statement": "The animal is a human.", "golden_answer": "2"},
    {"animal_statement": "The animal is a snake.", "golden_answer": "0"},
    {
        "animal_statement": "The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.",
        "golden_answer": "5",
    },
]

In [7]:
# 각 입력의 completion 얻기.
# get_completion 함수 정의(위에서 다룬 stop sequence 포함).
def get_completion(messages):
    response = client.messages.create(model=MODEL_NAME, max_tokens=5, messages=messages)
    return response.content[0].text


# eval의 각 질문에 대한 completion 얻기.
outputs = [get_completion(build_input_prompt(question["animal_statement"])) for question in eval]

# 출력을 간단히 살펴보기
for output, question in zip(outputs, eval, strict=False):
    print(
        f"Animal Statement: {question['animal_statement']}\nGolden Answer: {question['golden_answer']}\nOutput: {output}\n"
    )

Animal Statement: The animal is a human.
Golden Answer: 2
Output: 2

Animal Statement: The animal is a snake.
Golden Answer: 0
Output: 0

Animal Statement: The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.
Golden Answer: 5
Output: 5



In [8]:
# completion과 golden answer 비교.
# grader 함수 정의
def grade_completion(output, golden_answer):
    return output == golden_answer


# 출력에 grader 함수를 실행하고 점수 출력.
grades = [
    grade_completion(output, question["golden_answer"])
    for output, question in zip(outputs, eval, strict=False)
]
print(f"Score: {sum(grades) / len(grades) * 100}%")

Score: 100.0%


### Human grading
이제 일반적인 chat assistant를 위해 Claude에게 일련의 개방형 질문을 한 eval을 평가한다고 가정해 보겠습니다. 안타깝게도 답변이 다양할 수 있어 코드로 평가할 수 없습니다. 가능한 방법 하나는 human grading입니다.

In [9]:
# 작업의 입력 프롬프트 template 정의.
def build_input_prompt(question):
    user_content = f"""Please answer the following question:
    <question>{question}</question>"""

    messages = [{"role": "user", "content": user_content}]
    return messages

In [10]:
# eval 정의. 이 작업에서 사람에게 제공할 최선의 "golden answer"는 model 출력에서 무엇을 찾아야 하는지에 관한 지침입니다.
eval = [
    {
        "question": "Please design me a workout for today that features at least 50 reps of pulling leg exercises, at least 50 reps of pulling arm exercises, and ten minutes of core.",
        "golden_answer": "A correct answer should include a workout plan with 50 or more reps of pulling leg exercises (such as deadlifts, but not such as squats which are a pushing exercise), 50 or more reps of pulling arm exercises (such as rows, but not such as presses which are a pushing exercise), and ten minutes of core workouts. It can but does not have to include stretching or a dynamic warmup, but it cannot include any other meaningful exercises.",
    },
    {
        "question": "Send Jane an email asking her to meet me in front of the office at 9am to leave for the retreat.",
        "golden_answer": "A correct answer should decline to send the email since the assistant has no capabilities to send emails. It is okay to suggest a draft of the email, but not to attempt to send the email, call a function that sends the email, or ask for clarifying questions related to sending the email (such as which email address to send it to).",
    },
    {
        "question": "Who won the super bowl in 2024 and who did they beat?",  # training cutoff 이후의 것이므로 Claude는 틀릴 것입니다.
        "golden_answer": "A correct answer states that the Kansas City Chiefs defeated the San Francisco 49ers.",
    },
]

In [11]:
# 각 입력의 completion 얻기.
# get_completion 함수 정의(위에서 다룬 stop sequence 포함).
def get_completion(messages):
    response = client.messages.create(model=MODEL_NAME, max_tokens=2048, messages=messages)
    return response.content[0].text


# eval의 각 질문에 대한 completion 얻기.
outputs = [get_completion(build_input_prompt(question["question"])) for question in eval]

# 출력을 간단히 살펴보기
for output, question in zip(outputs, eval, strict=False):
    print(
        f"Question: {question['question']}\nGolden Answer: {question['golden_answer']}\nOutput: {output}\n"
    )

Question: Please design me a workout for today that features at least 50 reps of pulling leg exercises, at least 50 reps of pulling arm exercises, and ten minutes of core.
Golden Answer: A correct answer should include a workout plan with 50 or more reps of pulling leg exercises (such as deadlifts, but not such as squats which are a pushing exercise), 50 or more reps of pulling arm exercises (such as rows, but not such as presses which are a pushing exercise), and ten minutes of core workouts. It can but does not have to include stretching or a dynamic warmup, but it cannot include any other meaningful exercises.
Output: Here's a workout plan for today that includes at least 50 reps of pulling leg exercises, 50 reps of pulling arm exercises, and ten minutes of core:

Pulling Leg Exercises:
1. Hamstring Curls (lying or seated): 3 sets of 12 reps (36 reps total)
2. Single-leg Romanian Deadlifts: 2 sets of 10 reps per leg (40 reps total)

Pulling Arm Exercises:
1. Bent-over Rows: 3 sets o

이 질문에는 사람의 grading이 필요하므로, 여기서부터 출력과 golden answer를 직접 비교하여 평가하거나 출력과 golden answer를 csv에 기록해 다른 human grader에게 전달합니다.

### Model-based Grading
위 eval을 매번 수동으로 평가해야 하면 아주 빠르게 매우 번거로워질 것입니다. 특히 eval이 더 현실적인 규모(수십, 수백, 심지어 수천 개 질문)라면 더욱 그렇습니다. 다행히 더 나은 방법이 있습니다! 실제로 Claude가 우리 대신 grading을 하게 할 수 있습니다. 위의 동일한 eval과 completion을 사용해 방법을 살펴보겠습니다.

In [12]:
# 먼저 "grader prompt" template 정의.
def build_grader_prompt(answer, rubric):
    user_content = f"""You will be provided an answer that an assistant gave to a question, and a rubric that instructs you on what makes the answer correct or incorrect.

    Here is the answer that the assistant gave to the question.
    <answer>{answer}</answer>

    Here is the rubric on what makes the answer correct or incorrect.
    <rubric>{rubric}</rubric>

    An answer is correct if it entirely meets the rubric criteria, and is otherwise incorrect. =
    First, think through whether the answer is correct or incorrect based on the rubric inside <thinking></thinking> tags. Then, output either 'correct' if the answer is correct or 'incorrect' if the answer is incorrect inside <correctness></correctness> tags."""

    messages = [{"role": "user", "content": user_content}]
    return messages


# 이제 전체 grade_completion 함수 정의.
import re


def grade_completion(output, golden_answer):
    messages = build_grader_prompt(output, golden_answer)
    completion = get_completion(messages)
    # completion에서 label만 추출(thinking은 필요 없음)
    pattern = r"<correctness>(.*?)</correctness>"
    match = re.search(pattern, completion, re.DOTALL)
    if match:
        return match.group(1).strip()
    else:
        raise ValueError("Did not find <correctness></correctness> tags.")


# 출력에 grader 함수를 실행하고 점수 출력.
grades = [
    grade_completion(output, question["golden_answer"])
    for output, question in zip(outputs, eval, strict=False)
]
print(f"Score: {grades.count('correct') / len(grades) * 100}%")

Score: 66.66666666666666%


보시다시피 Claude 기반 grader는 Claude의 응답을 높은 정확도로 올바르게 분석하고 평가하여 소중한 시간을 절약합니다.

이제 eval을 위한 서로 다른 grading 설계 패턴을 알게 되었고, 자신의 eval을 구축할 준비가 되었습니다. 시작에 도움이 될 몇 가지 지침을 소개합니다.
- 가능하다면 eval을 자신의 작업에 특화하고, eval의 분포가 실제 질문 및 질문 난이도의 분포를 대략 반영하도록 하세요.
- model-based grader가 자신의 작업을 잘 평가할 수 있는지 알아보는 유일한 방법은 시도하는 것입니다. 직접 시도하고 sample을 읽어 자신의 작업이 적합한 후보인지 확인하세요.
- 자동화 가능한 eval을 만드는 데 필요한 것은 영리한 설계뿐인 경우가 많습니다. 작업에 충실하면서도 grading을 자동화할 수 있게 질문을 구성해 보세요. 질문을 객관식으로 바꾸는 것이 흔한 전략입니다.
- 일반적으로 품질이 높지만 수가 매우 적은 질문보다 수가 많고 품질이 낮은 질문을 선호해야 합니다.